Gold reproducibility notebook

- Uses the Silver pipeline (config + Zenodo download + deterministic seeds + PyTorch) and adds provenance capture (requirements-gold.txt, manifest.json with hashes, metrics, and git info).
- Configure `DATA_DOI`, `MODEL_DOI`, and filenames below.

In [ ]:
# Configuration and reproducibility helpers
from __future__ import annotations
import os, sys, json, hashlib, random, pathlib, time
from typing import Optional
import numpy as np

# Bronze DOIs used by default (copied from Silver)
DATA_DOI = os.environ.get("DATA_DOI", "10.5281/zenodo.17298664")
MODEL_DOI = os.environ.get("MODEL_DOI", "10.5281/zenodo.17298751")
DATA_FILENAME = os.environ.get("DATA_FILENAME", "simple_dataset.csv")
MODEL_FILENAME = os.environ.get("MODEL_FILENAME", "linear_model.pt")
ARTIFACTS_DIR = os.environ.get("ARTIFACTS_DIR", "artifacts")
DATA_DIR = os.environ.get("DATA_DIR", "Data")
VERBOSE = True

EXPECTED_DATA_SHA256 = os.environ.get("EXPECTED_DATA_SHA256", "")
EXPECTED_MODEL_SHA256 = os.environ.get("EXPECTED_MODEL_SHA256", "")

GLOBAL_SEED = int(os.environ.get("GLOBAL_SEED", "674"))
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# Utils
def ensure_dir(path: str) -> None:
    pathlib.Path(path).mkdir(parents=True, exist_ok=True)

def sha256_of_file(path: str) -> str:
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha.update(chunk)
    return sha.hexdigest()

def verify_hash(path: str, expected_sha256: str) -> bool:
    if not expected_sha256:
        return True
    actual = sha256_of_file(path)
    if VERBOSE:
        print(f"SHA256 for {path}: {actual}")
    return actual.lower() == expected_sha256.lower()

def zenodo_download(doi: str, filename: str, dest_dir: str) -> str:
    import urllib.request
    ensure_dir(dest_dir)
    record_id = doi.split(".")[-1].replace("zenodo/", "").replace("zenodo-", "").replace("zenodo", "").replace("/", "")
    url = f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"
    dest_path = os.path.join(dest_dir, filename)
    if VERBOSE:
        print(f"Downloading {url} -> {dest_path}")
    urllib.request.urlretrieve(url, dest_path)
    return dest_path

ensure_dir(ARTIFACTS_DIR)
ensure_dir(DATA_DIR)
print("Configuration loaded. Seeds set.")

   UserID  Age      City  Salary  Score
0       1   56   Chicago   91717  55.16
1       2   46   Chicago   95859  59.17
2       3   32  New York   71309  51.28
3       4   60   Chicago  108734  71.19
4       5   25   Chicago  115467  54.51


In [ ]:
# Torch seeding for determinism
import torch

# Seeds
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)


In [ ]:
import os, pandas as pd
# Load dataset from local Data/ or download from Zenodo
local_path = os.path.join(DATA_DIR, DATA_FILENAME)
if not os.path.exists(local_path):
    if not DATA_DOI:
        raise RuntimeError("DATA_DOI is empty and local data not found.")
    local_path = zenodo_download(DATA_DOI, DATA_FILENAME, DATA_DIR)
    if EXPECTED_DATA_SHA256 and not verify_hash(local_path, EXPECTED_DATA_SHA256):
        raise RuntimeError("Downloaded data hash mismatch; refusing to proceed.")
print(f"Using dataset at: {local_path}")
df = pd.read_csv(local_path)
print(df.head())


In [ ]:
# Preview loaded data
print(df.head())


In [ ]:
import os, pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

start_time = time.time()

# Preprocess
df_processed = pd.get_dummies(df, columns=['City'], drop_first=True)
X = df_processed.drop(['UserID', 'Score'], axis=1).values
y = df_processed['Score'].values.reshape(-1, 1)

# Deterministic split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=GLOBAL_SEED
)

# Torch tensors
X_train_t = torch.tensor(X_train.astype(np.float32))
y_train_t = torch.tensor(y_train.astype(np.float32))
X_test_t = torch.tensor(X_test.astype(np.float32))
y_test_t = torch.tensor(y_test.astype(np.float32))

# Simple linear model
class LinearRegressionModel(nn.Module):
    def __init__(self, input_size: int, output_size: int):
        super().__init__()
        self.linear = nn.Linear(input_size, output_size)
    def forward(self, x):
        return self.linear(x)

model = LinearRegressionModel(X_train.shape[1], 1)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4)

# Train
num_epochs = 10000
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    preds = model(X_train_t)
    loss = criterion(preds, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# Evaluate
model.eval()
with torch.no_grad():
    y_pred_t = model(X_test_t)

y_pred = y_pred_t.numpy()
y_true = y_test_t.numpy()

mse = mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print("--- Model Performance (PyTorch) ---")
print(f"MSE: {mse:.2f}")
print(f"R2: {r2:.2f}")

# Save model artifact
ensure_dir(ARTIFACTS_DIR)
model_path = os.path.join(ARTIFACTS_DIR, MODEL_FILENAME)
torch.save(model.state_dict(), model_path)
model_sha = sha256_of_file(model_path)
print(f"Saved torch state_dict to {model_path}")
print(f"Model SHA256: {model_sha}")

# Gold provenance: environment freeze and manifest
ensure_dir(ARTIFACTS_DIR)
req_path = os.path.join(ARTIFACTS_DIR, "requirements-gold.txt")
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
with open(req_path, "w") as f:
    f.write(freeze)
print(f"Saved environment to {req_path}")

# Git metadata
try:
    import subprocess as sp
    commit = sp.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    branch = sp.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip()
except Exception:
    commit = ""
    branch = ""

manifest = {
    "data": {
        "path": os.path.join(DATA_DIR, DATA_FILENAME),
        "sha256": sha256_of_file(os.path.join(DATA_DIR, DATA_FILENAME)) if os.path.exists(os.path.join(DATA_DIR, DATA_FILENAME)) else "",
        "doi": DATA_DOI,
    },
    "model": {
        "path": model_path,
        "sha256": model_sha,
        "doi": MODEL_DOI,
    },
    "metrics": {"mse": float(mse), "r2": float(r2)},
    "env": {
        "requirements": req_path,
        "python": sys.version,
        "numpy": np.__version__,
        "torch": torch.__version__,
    },
    "git": {"commit": commit, "branch": branch},
    "runtime_seconds": round(time.time() - start_time, 3),
}

man_path = os.path.join(ARTIFACTS_DIR, "manifest.json")
with open(man_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Saved manifest to {man_path}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

df_processed = pd.get_dummies(df, columns=['City'], drop_first=True)

X = df_processed.drop(['UserID', 'Score'], axis=1)
y = df_processed['Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=674)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Model Performance ---")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R²): {r2:.2f}")


print("\n--- Model Coefficients ---")

coeffs = pd.DataFrame(model.coef_, X_train.columns, columns=['Coefficient'])
print(coeffs)


--- Model Performance ---
Mean Squared Error (MSE): 29.99
R-squared (R²): 0.75

--- Model Coefficients ---
                  Coefficient
Age                  0.564729
Salary               0.000333
City_Houston        -2.968734
City_Los Angeles     8.420214
City_New York       12.329229
City_Phoenix        -8.096262
